In [40]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import LeaveOneOut, GridSearchCV, RandomizedSearchCV, KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from scipy.stats import pearsonr, spearmanr
from xgboost import XGBRegressor
import time
import matplotlib.pyplot as plt

In [41]:
# Custom graph visualization function
def regression_eval_plot(
    y_true,
    y_pred,
    title,
    xlabel="Actual",
    ylabel="Predicted",
    labels=None,
    outlier_z=2.5,
    figsize=(9, 9),
    save_path=None
):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    pearson_r, pearson_p = pearsonr(y_true, y_pred)
    spearman_r, spearman_p = spearmanr(y_true, y_pred)

    residuals = y_pred - y_true
    z_scores = (residuals - residuals.mean()) / residuals.std(ddof=1)
    outliers = np.where(np.abs(z_scores) > outlier_z)[0]

    fig, ax = plt.subplots(figsize=figsize)

    ax.scatter(
        y_true, y_pred,
        s=60, alpha=0.65,
        edgecolors="black", linewidth=0.6,
        label="Predictions"
    )

    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    ax.plot(
        [min_val, max_val],
        [min_val, max_val],
        "r--", lw=2, alpha=0.7,
        label="Perfect Prediction"
    )

    m, b = np.polyfit(y_true, y_pred, 1)
    ax.plot(
        y_true,
        m * y_true + b,
        "b-", lw=2, alpha=0.8,
        label=f"Best Fit (y={m:.2f}x+{b:.2f})"
    )

    for i in outliers:
        label = labels[i] if labels is not None else f"#{i}"
        ax.annotate(
            label,
            (y_true[i], y_pred[i]),
            xytext=(6, 6),
            textcoords="offset points",
            fontsize=9,
            color="darkred"
        )

    metrics_text = (
        f"Performance Metrics\n"
        f"────────────────────\n"
        f"R²: {r2:.4f}\n"
        f"RMSE: {rmse:.4f}\n"
        f"MAE: {mae:.4f}\n"
        f"Pearson r: {pearson_r:.4f}\n"
        f"Spearman ρ: {spearman_r:.4f}\n"
        f"N: {len(y_true)}\n"
        f"Outliers: {len(outliers)}"
    )

    ax.text(
        0.05, 0.95,
        metrics_text,
        transform=ax.transAxes,
        fontsize=11,
        va="top",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.85),
        family="monospace"
    )

    ax.set_xlabel(xlabel, fontsize=14, fontweight="bold")
    ax.set_ylabel(ylabel, fontsize=14, fontweight="bold")
    ax.set_title(title, fontsize=16, fontweight="bold", pad=18)

    ax.grid(True, linestyle="--", alpha=0.3)
    ax.legend(loc="lower right")
    ax.set_aspect("equal", adjustable="box")

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()

    return {
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "pearson_r": pearson_r,
        "pearson_p": pearson_p,
        "spearman_r": spearman_r,
        "spearman_p": spearman_p,
        "outliers": outliers
    }

In [42]:
# group drugs by mechanism

cell_wall_drugs = [
"Ampicillin",
"Cefaclor",
"Cycloserine",
"Ethambutol",
"Oxacillin",
"Vancomycin",
]

protein_synthesis_drugs = [
"Chloramphenicol",
"Azithromycin",
"Clarithromycin",
"Erythromycin",
"Doxycycline",
"Kanamycin",
"Linezolid",
"Minocycline",
"Roxithromycin",
"Streptomycin",
"Spectinomycin",
"Tetracycline",
]

nucleic_acid_drugs = [
"Ciprofloxacin",
"Levofloxacin",
"Moxifloxacin",
"Norfloxacin",
"Ofloxacin",
"Rifampicin",
]

antitubercular_drugs = [
"Bedaquiline",
"Capreomycin",
"Cycloserine",
"Delamanid",
"Ethambutol",
"Ethionamide",
"Isoniazid",
"PA-824 (Pretomanid)",
"Macozinone",
]

antibacterial_drugs = [
"Amikacin",
"Ampicillin",
"Azithromycin",
"Bedaquiline",
"Capreomycin",
"Cefaclor",
"Chloramphenicol",
"Ciprofloxacin",
"Clarithromycin",
"Clofazimine",
"Cycloserine",
"Delamanid",
"Doxycycline",
"Ethambutol",
"Erythromycin",
"Ethionamide",
"Fusidic acid",
"Isoniazid",
"Kanamycin",
"Levofloxacin",
"Linezolid",
"Minocycline",
"Moxifloxacin",
"Nitrofurantoin",
"Norfloxacin",
"Ofloxacin",
"Oxacillin",
"Rifampicin",
"Roxithromycin",
"Streptomycin",
"Spectinomycin",
"Sulfamethoxazole",
"Tetracycline",
"Trimethoprim",
"Vancomycin"
]

drug_groups = {
    "cell_wall_inhibitors": cell_wall_drugs,
    "protein_biosynthesis_inhibitors": protein_synthesis_drugs,
    "nucleic_acid_inhibitors": nucleic_acid_drugs,
    "antitubercular_drugs": antitubercular_drugs,
    "antibacterial_drugs": antibacterial_drugs,
}

In [43]:
# get training/testing drug interactions
# drug_interactions = pd.read_excel("all_but_rhoads_avg.xlsx")
drug_interactions = pd.read_excel("all_drugs_duplicates.xlsx")

drug_interactions.head(5)

,Drug_1,Drug_2,Drug_3,score
0,BDQ,CHLORAMPHENICOL,NaN,0.2137
1,BDQ,CLOFAZIMINE,NaN,-1.1613
2,BDQ,CYCLOSERINED,NaN,0.5646
3,BDQ,DELx,NaN,-0.2689
4,BDQ,ETA,NaN,-0.1714


In [44]:
# get all drug - target interactions for each drug in the training/testing set
ml_all = pd.read_csv("out.csv")
ml_all.head(5)

,Protein,AMIKACIN,AMP,AZITHROMYCIN,BDQ,CAP,CEF,CHLORAMPHENICOL,CIPROFLOXACIN,CLARYTHROMYCIN,...,SPECTINOMYCIN,SQ109,SUTx,TET,THZ (1hrMIC),TRZ,TUNICAMYCIN,VANCOMYCIN,VERAPAMIL,VERx
0,A0A089QRB9,5.587258e+05,1.399565e+05,3.600294e+05,68849.73116,4.719413e+05,1.612092e+05,6.421826e+05,30507.08200,2.775397e+05,...,2.829885e+05,5.808609e+05,221262.16440,1.950237e+05,1.890772e+06,30726.16403,4.928605e+05,67956.16584,55359.63472,55359.63472
1,I6WXK4,1.590979e+06,9.989999e+05,1.417508e+06,58576.56167,1.936968e+06,1.348820e+06,8.178893e+05,30648.22805,1.147954e+06,...,1.285868e+06,1.557001e+06,704274.93900,1.312658e+06,2.390992e+06,31826.43944,1.665023e+06,27530.24866,47098.24685,47098.24685
2,I6WZG6,6.264177e+05,1.195937e+06,1.058507e+06,54186.98233,1.287541e+06,1.211664e+06,3.264945e+05,24985.88472,9.802976e+05,...,8.366083e+05,9.514084e+05,512721.60620,1.094880e+06,7.664477e+05,19825.25500,1.186387e+06,29618.82852,79423.38532,79423.38532
3,I6X235,1.891681e+06,1.075159e+06,1.126003e+06,42390.45050,1.807572e+06,9.342417e+05,1.349153e+06,18627.88157,8.052585e+05,...,7.456942e+05,2.049744e+06,472636.63130,8.041869e+05,4.125644e+06,38897.21635,1.443836e+06,22641.50821,129071.61300,129071.61300
4,I6X8D2,5.308340e+04,1.078752e+05,2.185379e+05,65411.38455,1.313637e+05,8.765339e+04,2.223600e+05,10237.07463,1.259828e+05,...,9.912719e+04,1.590560e+05,62256.22142,1.142792e+05,7.167154e+05,37312.81417,9.410512e+04,40659.81937,82105.79162,82105.79162


In [45]:
# set a cutoff percentile for top drug - target hits
percentile = 20

In [46]:
# get drug names 
drug_names_ml = pd.read_excel('drugs_mtb.xlsx')
drug_names_ml.head(5)

,Full name,abbrev,SMILES
0,Amikacin,AMIKACIN,C1[C@@H]([C@H]([C@@H]([C@H]([C@@H]1NC(=O)[C@H]...
1,Ampicillin,AMP,CC1([C@@H](N2[C@H](S1)[C@@H](C2=O)NC(=O)[C@@H]...
2,Azithromycin,AZITHROMYCIN,CC[C@@H]1[C@@]([C@@H]([C@H](N(C[C@@H](C[C@@]([...
3,Bedaquiline,BDQ,CN(C)CC[C@@](C1=CC=CC2=CC=CC=C21)([C@H](C3=CC=...
4,Capreomycin,CAP,C[C@H]1C(=O)N[C@H](C(=O)N/C(=C/NC(=O)N)/C(=O)N...


In [47]:
# create the binarize function which will take the drug - target interactions and change them to 1, 0 depending on the percentile cutoff
# drug_target_interactions continuous scores for protein-ligand interactions)
# percentile (percentage of scores a particular score must be above to be considered a hit)
# drug_names (names of drugs that you want binary interaction scores for)
def binarize(drug_target_interactions: pd.DataFrame, drug_names: list, percentile: float):
    # get only drug columns but save the protein names
    dti = drug_target_interactions.iloc[:, 1:]
    drugs_all = dti.columns.tolist()
    proteins = drug_target_interactions['Protein'].values

    # get the top percentile hits
    p = dti.quantile(percentile / 100)
    binary_all = (dti < p).astype(int)

    # select only the drugs requested
    binary = binary_all[drug_names].to_numpy()
    
    return binary, proteins

binary, proteins = binarize(ml_all, drug_names_ml['abbrev'], percentile)

In [48]:
# calculate a sigma and delta score for each drug - drug interaction (method based on MAGENTA and INDIGO) 
# drug array must match the order and size of the binary matrix 
# rows_to_delete removes the drug-drug interactions that contain drugs not in the binary matrix (i.e. don't have available DTI data)
def create_sigma_delta(drug_interactions, binary, binary_drugs):
    
    # Stop if binary matrix doesn't have enough drugs
    if binary.shape[1] < len(binary_drugs):
        raise ValueError(
            f"Error: Binary matrix data does not contain enough information about all drugs.\n"
            f"We have numbers for {len(binary_drugs)} drugs but our input binary matrix has only {binary.shape[1]}"
        )
    
    # initialize the sigma/delta matrices
    num_interactions = drug_interactions.shape[0]
    num_samples = binary.shape[0]

    # sigma/delta scores are the size of drug interactions and number of proteins 
    sigma = np.zeros((num_interactions, num_samples))
    delta = np.zeros((num_interactions, num_samples))
    rows_to_delete = []

    binary_drugs_list = list(binary_drugs.values)

    for i in range(num_interactions):
        drugs = [drug_interactions.iloc[i, 0], drug_interactions.iloc[i, 1], drug_interactions.iloc[i, 2]]
        drugs_cleaned = [d for d in drugs if pd.notna(d) and str(d).lower() != 'nan']

        # if all drugs in the interaction are in binary dataset
        if all(d in binary_drugs_list for d in drugs_cleaned):
            # get scores for drugs
            logic = [binary_drugs_list.index(d) for d in drugs_cleaned]
            scores = binary[:, logic]
        
            # calculate sigma delta            
            sigma[i, :] = (np.sum(scores, axis=1) * 2 / scores.shape[1])
            delta[i, :] = (np.sum(scores > 0, axis=1) == 1).astype(int)         

        # drugs not in drug set add to rows to delete
        else:
            rows_to_delete.append(i)
            

    print(f"There are {len(rows_to_delete)} interaction scores that are not going to be considered.")
    binary_drugs_set = set(binary_drugs)
    interaction_drugs = pd.concat([
        drug_interactions['Drug_1'],
        drug_interactions['Drug_2'],
        drug_interactions['Drug_3'].dropna()
    ])

    interaction_drugs = interaction_drugs[interaction_drugs.astype(str).str.lower() != 'nan'].unique()
    missing_drugs = [d for d in interaction_drugs if d not in binary_drugs_set]

    print(f"There is/are {len(missing_drugs)} drug(s) that is/are not in the drug names dataset:")
    print(missing_drugs)

    if not rows_to_delete:
        rows_to_delete = [0]
    return sigma, delta, np.array(rows_to_delete)

sigma, delta, r2del = create_sigma_delta(drug_interactions, binary, drug_names_ml['abbrev'])

There are 6 interaction scores that are not going to be considered.
There is/are 6 drug(s) that is/are not in the drug names dataset:
['2169Uganda_rifampicin_resistant', 'HN878_rif_resistant', 'TKK_010025_ethionamide_resistant', 'TKK_010033_ethionamide_resistant', 'TKK_010040_ethionamide_resistant', 'TRS_OFX_resistant']


In [49]:
# Run ML algorithm to predict DDI. 
# Hold out assessment 

# hold out interations
n = 100

# remove drug interactions that contain drugs not in the dataset. This will only apply to limited datasets like chemogenomics that we must use as is
# if running stage 1 prior, the rows to delete will correspond to any compounds without SMILES structures etc. 
if r2del.size == 0:
    sig = sigma
    delt = delta
    DI = drug_interactions
else:
    sig = np.delete(sigma, r2del, axis=0)
    delt = np.delete(delta, r2del, axis=0)
    DI = drug_interactions.drop(index=r2del).reset_index(drop=True)

X = np.concatenate((sig, delt), axis=1)
y = DI['score']

In [ ]:
mse_scores = []
r2_scores = []
r_scores = []


param_grid = {
    'n_estimators': [300, 500],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    'alpha': [0, 1],     # L2 regularization term, the higher the tighter regularization, default 1
    'lambda': [1, 5],    # L1 regularization term, the higher the tighter regularization, default 0
    'gamma': [0, 0.5],      # gamma regularization term, the higher the tighter regularization, default 0
}

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42) # testsize 0.2

# set early stopping rounds to make sure the model will stop
# boosting iteration if no improvement for set rounds
xgb_model = XGBRegressor(random_state=42)
cv = KFold(n_splits=5, shuffle=True, random_state=42)
# Change scoring function base on you task
# you can change the verbose to 1 if the log is too much
grid_search = GridSearchCV(xgb_model, param_grid, cv=cv, 
                           scoring='neg_mean_squared_error',
                           verbose=1,
                           n_jobs=-1)

# Fit the GridSearchCV object to the training data
grid_search.fit(X_train, y_train)

# Print the best set of hyperparameters and the corresponding score
print("Best set of hyperparameters: ", grid_search.best_params_)
print("Best score: ", grid_search.best_score_)

best_model = grid_search.best_estimator_
best_params = grid_search.best_params_


Fitting 5 folds for each of 384 candidates, totalling 1920 fits


In [ ]:
y_pred_test = best_model.predict(X_test)

print("MSE:", mean_squared_error(y_test, y_pred_test))
print("R2:", r2_score(y_test, y_pred_test))
print("Pearson R:", pearsonr(y_test, y_pred_test)[0])


In [ ]:
mse_scores = []
r2_scores = []
r_scores = []

start = time.perf_counter()

for i in range(n):
    # split the data for hold out
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=None)

    # create and train the model
    ft = XGBRegressor(**best_params)

    ft.fit(X_train, y_train)

    # predict for test set
    y_pred = ft.predict(X_test)

    # calculate metrics
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    r = pearsonr(y_test, y_pred)[0]

    # store metrics for each iteration
    mse_scores.append(mse)
    r2_scores.append(r2)
    r_scores.append(r)

end = time.perf_counter()

# Summary statistics
print(f"Mean MSE over {n} iterations: {np.mean(mse_scores):.4f} ± {np.std(mse_scores):.4f}")
print(f"Mean R2 over {n} iterations: {np.mean(r2_scores):.4f} ± {np.std(r2_scores):.4f}")

print(f"Mean Pearson R over {n} iterations: {np.mean(r_scores):.4f} ± {np.std(r_scores):.4f}")
print(f"Time Elapsed: {end-start:.2f} seconds")

plt.figure(figsize=(8,6))
plt.hist(r_scores, bins=20)
plt.axvline(np.mean(r_scores), linestyle='--')
# more formal title: Distribution of Predictive Performance (Pearson’s r) Across 100 Monte Carlo Cross-Validation Iterations
plt.title("Pearson Correlation (r) Over 100 Random Train/Test Splits")
plt.xlabel("Pearson r")
plt.ylabel("Frequency")
plt.show()
    

In [ ]:
# Leave one out cross validation for single interactions
# The loop does a LOOCV for every single interaction then graphs all of them

X = np.array(X)
y = np.array(y)

loo = LeaveOneOut()
y_true, y_pred = [], []
start = time.perf_counter()

for train_index, test_index in loo.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    ft = XGBRegressor(**best_params)
    # ft = GradientBoostingRegressor()
    ft.fit(X_train,y_train)
    
    y_pred.append(ft.predict(X_test)[0])
    y_true.append(y_test[0])

end = time.perf_counter()
r2 = r2_score(y_true, y_pred)
mse = mean_squared_error(y_true, y_pred)
pearson_r = pearsonr(y_true, y_pred)[0]

print(f"Mean MSE: {mse:.4f}")
print(f"R² Score: {r2:.4f}")
print(f"Pearson R: {pearson_r:.4f}")
print(f"Time Elapsed: {end - start:.2f} seconds")

regression_eval_plot(
    y_true=y_true,
    y_pred=y_pred,
    title="LOOCV for Single Interactions Predicted vs Actual",
    xlabel="Actual Interaction Score",
    ylabel="Predicted Interaction Score",
    labels=DI.loc[:, ["Drug_1", "Drug_2"]]
            .astype(str)
            .agg("+".join, axis=1)
            .values
)

In [ ]:
# Leave one out cross validation for certain individual drugs (& all their interactions)

LOOCV_drug_r = []
y = DI['score'].values

all_drugs = pd.concat([DI['Drug_1'], DI['Drug_2']]).unique()

for drug in all_drugs:
    drug_included = (DI["Drug_1"] == f"{drug}") | (DI["Drug_2"] == f"{drug}")

    mask = drug_included.values
    if mask.sum() < 10:
        continue

    X_train, X_test = X[~drug_included], X[drug_included]
    y_train, y_test = y[~drug_included], y[drug_included]

    ft = XGBRegressor(**best_params)
    ft.fit(X_train, y_train)

    y_pred = ft.predict(X_test)

    r = pearsonr(y_test, y_pred)[0]
    # print(f"Pearson R: {r:.3f}")
    LOOCV_drug_r.append(r)

    regression_eval_plot(
        y_true=y_test,
        y_pred=y_pred,
        title=f"LOOCV for {drug} Predicted vs Actual",
        xlabel="Actual Interaction Score",
        ylabel="Predicted Interaction Score",
        labels=DI.loc[drug_included, ["Drug_1", "Drug_2"]].astype(str).agg("+".join, axis=1).values
    )

r_vals = np.array(LOOCV_drug_r)
plt.figure()
plt.hist(r_vals, bins=20)
plt.xlabel("Pearson r")
plt.ylabel("Number of Drugs")
plt.title("Pearson R for LOOCV of each drug")
plt.show()

print(LOOCV_drug_r)
avg_r = (sum(LOOCV_drug_r)*1.0)/len(LOOCV_drug_r)
print(f"Average R: {avg_r}")

In [ ]:
results = {}

for group_name, group_list in drug_groups.items():

    group_abbrevs = [d for d in drug_names_ml['abbrev'] if d.upper() in [c.upper() for c in group_list]]
    
    mask = DI['Drug_1'].isin(group_abbrevs) | DI['Drug_2'].isin(group_abbrevs)
    group_interactions = DI[mask] 
    
    if group_interactions.shape[0] > 0:
        group_indices = group_interactions.index.values
        X_group = X[group_indices]
        y_group = group_interactions['score'].values

        loo = LeaveOneOut()
        y_true_group, y_pred_group = [], []
        for train_idx, test_idx in loo.split(X_group):
            X_train, X_test = X_group[train_idx], X_group[test_idx]
            y_train, y_test = y_group[train_idx], y_group[test_idx]

            ft = XGBRegressor(**best_params)
            ft.fit(X_train, y_train)
            y_pred_group.append(ft.predict(X_test)[0])
            y_true_group.append(y_test[0])

        r2_group = r2_score(y_true_group, y_pred_group)
        mse_group = mean_squared_error(y_true_group, y_pred_group)
        pearson_r_group = pearsonr(y_true_group, y_pred_group)[0]

        results[group_name] = {
            "mse": mse_group,
            "r2": r2_group,
            "pearson_r": pearson_r_group,
            "n": len(y_true_group)
        }

        print(f"{group_name} LOOCV - Mean MSE: {mse_group:.4f}")
        print(f"{group_name} LOOCV - R² Score: {r2_group:.4f}")
        print(f"{group_name} LOOCV - Pearson R: {pearson_r_group:.4f}")

        regression_eval_plot(
        y_true=y_true_group,
        y_pred=y_pred_group,
        title=f"LOOCV for {group_name.replace('_',' ').title()}\nPredicted vs Actual",
        xlabel="Actual Score",
        ylabel="Predicted Score",
        labels=group_interactions[["Drug_1", "Drug_2"]]
            .astype(str)
            .agg("+".join, axis=1)
            .values
        )

    else:
        print(f"No interactions found with 2 {group_name}.")
